In [2]:
import pandas as pd
from pathlib import Path

LOADING DATA

In [ ]:
# parquet file storage for more compressed representation vs JSON
data_path = Path.cwd() / "raw_overflow_data.parquet"
print(f"Data exist: {data_path.exists()}")
print(f"Check if is a file: {data_path.is_file()}")

Data exist: True
Check if is a file: True


In [4]:
# extracting the data with pandas
df = pd.read_parquet(data_path)
df.head()

,topic,question_title,question_body,answer_body
0,ABSTRACT CLASSES,Interfaces vs Types in TypeScript,What is the difference between these statement...,Update March 2021: The newer TypeScript Handbo...
1,ABSTRACT CLASSES,Can an abstract class have a constructor?,Can an abstract class have a constructor?\n\nI...,"Yes, an abstract class can have a constructor...."
2,ABSTRACT CLASSES,How to determine if a type implements an inter...,Does reflection in C# offer a way to determine...,You have a few choices:\n\ntypeof(IMyInterface...
3,ABSTRACT CLASSES,C# Interfaces. Implicit implementation versus ...,What are the differences in implementing inter...,Implicit is when you define your interface via...
4,ABSTRACT CLASSES,Why can't static methods be abstract in Java?,The question is in Java why can't I define an ...,The abstract annotation to a method indicates ...


In [5]:
print(f"Null values check.")
df.isna().sum()

Null values check.


topic             0
question_title    0
question_body     0
answer_body       0
dtype: int64

In [6]:
# all topics
topics = df['topic'].unique()
topics

<ArrowStringArray>
[            'ABSTRACT CLASSES',                  'AGGREGATION',
                   'ALGORITHMS',                       'ARRAYS',
                 'BINARY TREES',        'C CHARACTER FUNCTIONS',
                  'COMPOSITION',           'DESIGN AND TESTING',
                'ENCAPSULATION',                 'GRAPH THEORY',
                  'HASH TABLES',                  'INHERITANCE',
                  'LINKED LIST',                         'MAPS',
                      'MERGING', 'MINIMAL AND COMPLETE CLASSES',
               'MULTIWAY TREES',                          'OOP',
                   'PARAMETERS',                     'POINTERS',
                       'QUEUES',                      'RECORDS',
                    'SEARCHING',                         'SETS',
                      'SORTING',                       'STACKS',
                      'STRINGS',                        'TREES',
   'TWO DIMENSIONAL STRUCTURES',                      'VECTORS']
Length

In [7]:
# 3000 was the max amount of data pulled per topic, trying to see if there's undersampled classes

def topics_under_size(df, counts=3000):
    
    topic_counts = df.loc[:,'topic'].value_counts()
    topic_counts_filter = topic_counts < counts
    return topic_counts[topic_counts_filter].sort_values()

topics_under_size(df)

topic
MINIMAL AND COMPLETE CLASSES     611
MULTIWAY TREES                   670
COMPOSITION                      695
ENCAPSULATION                    841
GRAPH THEORY                    1319
AGGREGATION                     1477
HASH TABLES                     2119
Name: count, dtype: int64

In [8]:
# we can fix some of these with mapping close topics together.
topic_map = {
    'COMPOSITION' : 'OOP_CONCEPTS',
    'AGGREGATION' : 'OOP_CONCEPTS',
    'ENCAPSULATION' : 'OOP_CONCEPTS',
    'MINIMAL AND COMPLETE CLASSES' : 'OOP_CONCEPTS',
    'GRAPH THEORY' : 'ADVANCED_TREES_GRAPHS',
    'MULTIWAY TREES' : 'ADVANCED_TREES_GRAPHS',
}

# while we can't get all classes to 3k we can get close
# during training, we can class weights to give give classes equal influences during training
mapped_df = df.copy()
mapped_df['topic'] = df['topic'].replace(topic_map)

topics_under_size(mapped_df)

topic
ADVANCED_TREES_GRAPHS    1989
HASH TABLES              2119
Name: count, dtype: int64

VISUALIZING DATA (KINDA)

In [9]:

def random_sample_topics(df, number_per_topic=2):
    
    # finding the count for smallest group
    smallest_topic = df['topic'].value_counts().min()
        
    # randomly sample 
    random_samples = df.groupby('topic').sample(n=number_per_topic, random_state=42)

    return random_samples.drop(columns=['question_title'])
    
random_sample_topics(mapped_df)

,topic,question_body,answer_body
1801,ABSTRACT CLASSES,I'm trying to serialize my data structures in ...,This is a little bit weird but I'm going to an...
1190,ABSTRACT CLASSES,Playground link: http://play.golang.org/p/Ebf5...,"text/template is special casing interface{}, s..."
21949,ADVANCED_TREES_GRAPHS,Given two nodes A and B from a directed JUNG g...,I found a solution myself.\n\nMy problem has t...
21584,ADVANCED_TREES_GRAPHS,The classic travelling salesman problem says y...,You can just replace the distances with the sh...
6173,ALGORITHMS,The binary search is highly efficient for unif...,There's a deep connection between binary searc...
5063,ALGORITHMS,I have some theoretical/practical problem and ...,I am going to try and first formalize the prob...
8909,ARRAYS,Starting with the following code...\n\nbyte fo...,The JLS (§5.2) has special rules for assignmen...
7603,ARRAYS,Will this work for testing whether a value at ...,"Conceptually, arrays in JavaScript contain arr..."
10781,BINARY TREES,I am trying to create a deep copy of my binary...,You're not performing a deep copy of your root...
10660,BINARY TREES,I need a sorted set of objects and am currentl...,"You're defining one criteria to compare, but y..."


NOTE: Due to the nature of our data (heavily code-based), common cleaning techniques can serve more to hinder than help. According to the average sample, there are a lot of new lines. These are likely the things that need the most attention.

In [10]:
import html

cleaned_df = mapped_df.copy()

# decode html code back into regular characters
cleaned_df = cleaned_df.apply(html.unescape)

# if the string pattern matches newline or carriage return (\r) ONE or MORE time, replace with space
cleaned_df = cleaned_df.replace(r'[\n\r]+', ' ', regex=True)

In [11]:
random_sample_topics(cleaned_df)

,topic,question_body,answer_body
1801,ABSTRACT CLASSES,I'm trying to serialize my data structures in ...,This is a little bit weird but I'm going to an...
1190,ABSTRACT CLASSES,Playground link: http://play.golang.org/p/Ebf5...,"text/template is special casing interface{}, s..."
21949,ADVANCED_TREES_GRAPHS,Given two nodes A and B from a directed JUNG g...,I found a solution myself. My problem has the ...
21584,ADVANCED_TREES_GRAPHS,The classic travelling salesman problem says y...,You can just replace the distances with the sh...
6173,ALGORITHMS,The binary search is highly efficient for unif...,There's a deep connection between binary searc...
5063,ALGORITHMS,I have some theoretical/practical problem and ...,I am going to try and first formalize the prob...
8909,ARRAYS,Starting with the following code... byte foo =...,The JLS (§5.2) has special rules for assignmen...
7603,ARRAYS,Will this work for testing whether a value at ...,"Conceptually, arrays in JavaScript contain arr..."
10781,BINARY TREES,I am trying to create a deep copy of my binary...,You're not performing a deep copy of your root...
10660,BINARY TREES,I need a sorted set of objects and am currentl...,"You're defining one criteria to compare, but y..."


In [12]:
topic_label_map = {key : value for value, key in enumerate(cleaned_df['topic'].value_counts().index)}
topic_label_map

{'OOP_CONCEPTS': 0,
 'ABSTRACT CLASSES': 1,
 'ALGORITHMS': 2,
 'ARRAYS': 3,
 'BINARY TREES': 4,
 'C CHARACTER FUNCTIONS': 5,
 'DESIGN AND TESTING': 6,
 'INHERITANCE': 7,
 'LINKED LIST': 8,
 'MAPS': 9,
 'MERGING': 10,
 'OOP': 11,
 'PARAMETERS': 12,
 'POINTERS': 13,
 'QUEUES': 14,
 'RECORDS': 15,
 'SEARCHING': 16,
 'SETS': 17,
 'SORTING': 18,
 'STACKS': 19,
 'STRINGS': 20,
 'TREES': 21,
 'TWO DIMENSIONAL STRUCTURES': 22,
 'VECTORS': 23,
 'HASH TABLES': 24,
 'ADVANCED_TREES_GRAPHS': 25}

In [13]:
# label mapping for ML work
cleaned_df['label'] = cleaned_df['topic'].map(topic_label_map)

# seeing the labels worked as intended (all true means nothing was mislabled)
cleaned_df['topic'].value_counts().values == cleaned_df['label'].value_counts().values

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True])

In [23]:
# saving to parquet
save_path = Path.cwd() / "train_ready.parquet"
cleaned_df.to_parquet(path=save_path)

if save_path.is_file():
    
    try:
        test = pd.read_parquet(save_path)
        
        print('If all zero, then file integrity is good.')
        print((test != cleaned_df).sum())
        
        print(f'Successfully saved: {save_path}')
        
    except Exception as e:
        print('Save Failed.', {e})
else:
    print('Save failed.')

If all zero, then file integrity is good.
topic             0
question_title    0
question_body     0
answer_body       0
label             0
dtype: int64
Successfully saved: c:\All University Materials\Project\ICT304-project\ai\ai_tools\data_preprocessing\train_ready.parquet
